<a href="https://colab.research.google.com/github/Eswar2005-Karanam/blog/blob/main/Text_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression

# ==========================================
# PART 1: TEXT PREPROCESSING TOOLKIT
# ==========================================
print("=" * 65)
print(f"{'PART 1: CLASSIC TEXT PREPROCESSING & VECTORIZATION':^65}")
print("=" * 65)

def preprocess_text(text):
    """
    Cleans raw text: converts to lowercase, strips non-alphabet characters,
    tokenizes, and removes standard English stop words.
    """
    # 1. Lowercase + strip non-letters
    clean = re.sub(r"[^a-z\s]", "", str(text).lower())

    # 2. Tokenize by whitespace
    tokens = clean.split()

    # 3. Remove stop words
    filtered_tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]

    return filtered_tokens

# Demo paragraph preprocessing
sample_text = "Natural Language Processing is AMAZING, in 2026!"
cleaned_tokens = preprocess_text(sample_text)

print(f"Original Text  : \"{sample_text}\"")
print(f"Cleaned Tokens : {cleaned_tokens}\n")

# --- 1. Bag-of-Words Matrix Representation ---
sents = ["i love nlp", "nlp is fun", "i love ai"]
cv = CountVectorizer()
bow_matrix = cv.fit_transform(sents)

df_bow = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out(), index=[f"Sent {i+1}" for i in range(len(sents))])
print("--- Bag-of-Words (CountVectorizer Matrix) ---")
print(df_bow)
print()

# --- 2. TF-IDF Matrix Representation ---
docs = ["the cat sat", "the dog ran", "the bird flew"]
tv = TfidfVectorizer()
tfidf_matrix = tv.fit_transform(docs)

df_tfidf = pd.DataFrame(tfidf_matrix.toarray().round(2), columns=tv.get_feature_names_out(), index=[f"Doc {i+1}" for i in range(len(docs))])
print("--- TF-IDF Matrix Representation ---")
print(df_tfidf)
print("\nNotice how common words ('the') get 0.0 weight across all documents, while rare words ('cat', 'dog', 'bird') receive high weights.\n")


# ==========================================
# PART 2: MODERN NLP (Hugging Face Transformers)
# ==========================================
print("=" * 65)
print(f"{'PART 2: HUGGINGFACE TRANSFORMER PIPELINES':^65}")
print("=" * 65)

try:
    from transformers import pipeline

    # 1. Zero-shot Sentiment Analysis Pipeline
    print("📥 Loading Sentiment Analysis Transformer...")
    clf = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
    sample_texts = ["I love this course!", "Worst purchase ever."]
    transformer_results = clf(sample_texts)

    print("\n--- Transformer Sentiment Predictions ---")
    for text, res in zip(sample_texts, transformer_results):
        print(f"Text  : \"{text}\"")
        print(f"Result: {res['label']} (Confidence: {res['score']*100:.2f}%)\n")

    # 2. Translation Pipeline
    print("📥 Loading Translation Transformer (English to French)...")
    translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
    en_text = "Artificial intelligence is amazing."
    translation_result = translator(en_text)
    print("\n--- Translation Task ---")
    print(f"EN: '{en_text}'")
    print(f"FR: '{translation_result[0]['translation_text']}'\n")

except Exception as e:
    print(f"⚠️ Transformer pipeline notice: {e}")
    print("Skipping Hugging Face execution (Classic NLP toolkit sections remain fully functional).\n")


# ==========================================
# PART 3: INTEGRATIVE REVIEW ANALYZER (Section 17.5)
# ==========================================
print("=" * 65)
print(f"{'PART 3: INTEGRATIVE REVIEW ANALYZER (End-to-End Pipeline)':^65}")
print("=" * 65)

class ReviewAnalyzer:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.model = LogisticRegression(max_iter=1000)

    def _preprocess_doc(self, text):
        """Preprocesses text and rejoins clean tokens into a normalized string."""
        tokens = preprocess_text(text)
        return " ".join(tokens)

    def train(self, documents, labels):
        # 1. Clean and preprocess all input documents
        processed_docs = [self._preprocess_doc(doc) for doc in documents]

        # 2. Fit TF-IDF Vectorizer
        X_vec = self.vectorizer.fit_transform(processed_docs)

        # 3. Fit Model Classifier
        self.model.fit(X_vec, labels)
        print(" SUCCESS: ReviewAnalyzer trained successfully.")

    def predict(self, new_reviews):
        processed_new = [self._preprocess_doc(r) for r in new_reviews]
        vec_new = self.vectorizer.transform(processed_new)
        preds = self.model.predict(vec_new)
        probs = self.model.predict_proba(vec_new)

        print("\n" + "-" * 65)
        print(f"{'INTEGRATIVE INFERENCE RESULTS':^65}")
        print("-" * 65)
        for rev, pred, prob in zip(new_reviews, preds, probs):
            confidence = max(prob) * 100
            tag = "🟢 [POSITIVE]" if pred == "positive" else "🔴 [NEGATIVE]"
            print(f"Input Review : \"{rev}\"")
            print(f"Predicted    : {tag} (Confidence: {confidence:.2f}%)")
            print("-" * 65)

# Balanced training dataset for Integrative Analyzer
train_reviews = [
    "This product is outstanding, works fast and smooth!",
    "Great item, highly satisfied with the build quality.",
    "Very happy with this purchase, worth every penny.",
    "Terrible experience, broke within one day of light use.",
    "Waste of money, completely useless product.",
    "Horrible service and poor quality control."
]
train_labels = ["positive", "positive", "positive", "negative", "negative", "negative"]

# Instantiate, train, and test integrative analyzer
analyzer = ReviewAnalyzer()
analyzer.train(train_reviews, train_labels)

# In-memory test inputs
test_reviews = [
    "Extremely fast and amazing quality!",
    "Waste of money, very bad performance."
]
analyzer.predict(test_reviews)

       PART 1: CLASSIC TEXT PREPROCESSING & VECTORIZATION        
Original Text  : "Natural Language Processing is AMAZING, in 2026!"
Cleaned Tokens : ['natural', 'language', 'processing', 'amazing']

--- Bag-of-Words (CountVectorizer Matrix) ---
        ai  fun  is  love  nlp
Sent 1   0    0   0     1    1
Sent 2   0    1   1     0    1
Sent 3   1    0   0     1    0

--- TF-IDF Matrix Representation ---
       bird   cat   dog  flew   ran   sat   the
Doc 1  0.00  0.65  0.00  0.00  0.00  0.65  0.39
Doc 2  0.00  0.00  0.65  0.00  0.65  0.00  0.39
Doc 3  0.65  0.00  0.00  0.65  0.00  0.00  0.39

Notice how common words ('the') get 0.0 weight across all documents, while rare words ('cat', 'dog', 'bird') receive high weights.

            PART 2: HUGGINGFACE TRANSFORMER PIPELINES            
📥 Loading Sentiment Analysis Transformer...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]


--- Transformer Sentiment Predictions ---
Text  : "I love this course!"
Result: POSITIVE (Confidence: 99.99%)

Text  : "Worst purchase ever."
Result: NEGATIVE (Confidence: 99.98%)

📥 Loading Translation Transformer (English to French)...


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

⚠️ Transformer pipeline notice: "Unknown task translation_en_to_fr, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"
Skipping Hugging Face execution (Classic NLP toolkit sections remain fully functional).

    PART 3: INTEGRATIVE REVIEW ANALYZER (End-to-End Pipeline)    
 SUCCESS: ReviewAnalyzer trained successfully.

-----------------------------------------------------------------
                  INT